 `pip3 install scapy` + root/admin pentru demo
- Linux/Mac: `sudo jupyter notebook`
- Windows: Jupyter ca Administrator + Npcap instalat

https://npcap.com/


---
## Demo IP Spoofing

Câmpul Source Address din IP header nu e verificat de nimeni pe traseu.
Scapy ne lasă să punem orice adresă dorim acolo.

Asta e baza SYN flood, atacurilor de amplificare, și impersonării.

**Necesită root/admin. Rulează DOAR pe propria rețea/mașină.**


In [1]:
!pip3 install scapy


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from scapy.all import IP, UDP, Raw, send
import socket

MY_REAL_IP = socket.gethostbyname(socket.gethostname())
print(f"IP-ul meu real: {MY_REAL_IP}")
print()

#Construim un pachet cu IP sursă falsificat
#Nota: trimitem la loopback (127.0.0.1) ca să nu ieșim în rețea
pachet_normal = IP(src=MY_REAL_IP, dst="127.0.0.1") /                 UDP(sport=1234, dport=5678) /                 Raw(load=b"pachet normal")

pachet_spoofed = IP(src="1.2.3.4", dst="127.0.0.1") /                  UDP(sport=1234, dport=5678) /                  Raw(load=b"pachet cu IP fals")

print("Pachet normal:")
print(f"  src={pachet_normal[IP].src}  dst={pachet_normal[IP].dst}")
print()
print("Pachet spoofed:")
print(f"  src={pachet_spoofed[IP].src}  dst={pachet_spoofed[IP].dst}")
print()
print("Scapy pune pur și simplu ce îi spunem în câmpul src.")
print("Niciun router nu verifică dacă acea adresă ne aparține cu adevărat.")


IP-ul meu real: 192.168.1.138

Pachet normal:
  src=192.168.1.138  dst=127.0.0.1

Pachet spoofed:
  src=1.2.3.4  dst=127.0.0.1

Scapy pune pur și simplu ce îi spunem în câmpul src.
Niciun router nu verifică dacă acea adresă ne aparține cu adevărat.


In [3]:
# Să vedem cum arată cele două pachete byte cu byte
import struct, socket as sock

def show_ip_src(pkt_bytes):
    # bytes 12-16 = IP sursă
    src = pkt_bytes[26:30]  # 14 bytes Ethernet header + 12 bytes IP = 26
    return sock.inet_ntoa(src)

# Construim pachetele și le serializăm
from scapy.layers.l2 import Ether
normal_bytes  = bytes(Ether()/pachet_normal)
spoofed_bytes = bytes(Ether()/pachet_spoofed)

print("Bytes raw, zona Source IP (bytes 26-30 din frame Ethernet):")
print()
print(f"  Normal:  {normal_bytes[26:30].hex()}  = {show_ip_src(normal_bytes)}")
print(f"  Spoofed: {spoofed_bytes[26:30].hex()}  = {show_ip_src(spoofed_bytes)}")
print()
print("Câmpul e identic structural, doar valoarea diferă.")
print("Un router care forwardează pachetul nu face nicio verificare.")


Bytes raw, zona Source IP (bytes 26-30 din frame Ethernet):

  Normal:  c0a8018a  = 192.168.1.138
  Spoofed: 01020304  = 1.2.3.4

Câmpul e identic structural, doar valoarea diferă.
Un router care forwardează pachetul nu face nicio verificare.


In [4]:
# De ce contează în practică

print("Scenarii de atac bazate pe IP spoofing:")
print()

scenarii = [
    ("SYN Flood",
     "Trimiți mii de SYN cu IP-uri random falsificate."
     "  Serverul alocă memorie pentru fiecare și trimite SYN-ACK la nimeni."
     "  Conexiunile half-open umplu tabelul — denial of service."),

    ("ICMP Amplification (Smurf)",
     "Trimiți ICMP Echo Request cu IP sursă = victima la adresa broadcast."
     "  Toți din rețea trimit Echo Reply la victimă simultan."
     "  Amplificare: 1 pachet trimis → N pachete primite de victimă."),

    ("DNS Amplification",
     "Trimiți DNS query cu IP sursă = victima la un resolver deschis."
     "  Răspunsul DNS (mult mai mare decât query-ul) se duce la victimă."
     "  Factor de amplificare tipic: 50×-70×."),

    ("Impersonare",
     "Trimiți pachete aparent de la un IP de încredere (ex: serverul de autentificare)."
     "  Victima primește pachete care par legitime."
     "  De aceea autentificarea nu se poate baza pe IP sursă."),
]

for titlu, desc in scenarii:
    print(f"  ► {titlu}")
    print(f"    {desc}")
    print()

print("Mitigare — BCP38 / RFC 2827:")
print("  ISP-urile ar trebui să filtreze la ieșire pachetele cu")
print("  IP sursă care nu aparțin subrețelei clientului.")
print("  Din păcate, adoptarea e incompletă la nivel global.")


Scenarii de atac bazate pe IP spoofing:

  ► SYN Flood
    Trimiți mii de SYN cu IP-uri random falsificate.  Serverul alocă memorie pentru fiecare și trimite SYN-ACK la nimeni.  Conexiunile half-open umplu tabelul — denial of service.

  ► ICMP Amplification (Smurf)
    Trimiți ICMP Echo Request cu IP sursă = victima la adresa broadcast.  Toți din rețea trimit Echo Reply la victimă simultan.  Amplificare: 1 pachet trimis → N pachete primite de victimă.

  ► DNS Amplification
    Trimiți DNS query cu IP sursă = victima la un resolver deschis.  Răspunsul DNS (mult mai mare decât query-ul) se duce la victimă.  Factor de amplificare tipic: 50×-70×.

  ► Impersonare
    Trimiți pachete aparent de la un IP de încredere (ex: serverul de autentificare).  Victima primește pachete care par legitime.  De aceea autentificarea nu se poate baza pe IP sursă.

Mitigare — BCP38 / RFC 2827:
  ISP-urile ar trebui să filtreze la ieșire pachetele cu
  IP sursă care nu aparțin subrețelei clientului.
  Din p

In [5]:
from scapy.all import conf
print(conf.iface)

\Device\NPF_{50CE922A-448F-4F5B-90E4-173831DFBB94}


In [6]:
# Demonstrație live: trimitem un pachet spoofed și îl capturăm
# ca să vedem că IP-ul fals apare exact cum l-am setat

from scapy.all import sniff, send, IP, UDP, Raw
import threading, time

FAKE_SRC   = "42.42.42.42"
LOCAL_PORT = 19999
captured   = []

def sniffer():
    def handler(pkt):
        if IP in pkt and UDP in pkt:
            if pkt[UDP].dport == LOCAL_PORT:
                captured.append(pkt)
    sniff(filter=f"udp port {LOCAL_PORT}", prn=handler,
          store=False, timeout=3, iface=conf.iface)

# pornim sniffer în background
t = threading.Thread(target=sniffer)
t.start()
time.sleep(1)

# trimitem pachetul spoofed
pkt = IP(src=FAKE_SRC, dst=MY_REAL_IP) /       UDP(sport=9999, dport=LOCAL_PORT) /       Raw(load=b"sunt un pachet fals")
send(pkt, verbose=False)

t.join()

if captured:
    p = captured[0]
    print(f"Pachet capturat:")
    print(f"  IP sursă în packet:      {p[IP].src}")
    print(f"  IP sursă real (al meu):  {MY_REAL_IP}")
    print(f"  Payload:                 {p[Raw].load}")
    print()
    print(f"tcpdump vede IP sursă = {p[IP].src}, nu {MY_REAL_IP}.")
    print("Asta e tot ce e IP spoofing.")
else:
    print("(rulează cu sudo pentru a putea trimite și captura pachete raw)")


(rulează cu sudo pentru a putea trimite și captura pachete raw)
